## Loading Packages

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import plotly.express as px

## Loading the Data

In [5]:
df = pd.read_csv('https://raw.githubusercontent.com/hovhannisyan91/data_analytics_with_python/refs/heads/main/data/cohort/cohort_analysis.csv',
                 parse_dates=['acquisition_date','cancellation_month']
                 )
df.head()

,user_id,acquisition_date,cancellation_month,gender,marital_status,age,income_segment,country,channel,campaign_id,device_type,plan_type
0,1,2024-07-01,2025-02-01,Male,Married,31,Medium,Germany,Paid Ads,Paid Ads_C,iOS,Standard
1,2,2024-04-01,2024-05-01,Male,Single,54,Premium,Netherlands,Referral,Referral_B,iOS,Standard
2,3,2024-05-01,2024-07-01,Male,Single,34,Medium,Poland,Paid Ads,Paid Ads_A,Android,Standard
3,4,2024-07-01,NaT,Male,Married,38,High,Belgium,Organic,Organic_C,Android,Standard
4,5,2024-03-01,2024-04-01,Male,Single,25,Low,Sweden,Paid Ads,Paid Ads_A,Android,Basic


In [6]:
df.info()
# check cancellation_month

<class 'pandas.DataFrame'>
RangeIndex: 12000 entries, 0 to 11999
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   user_id             12000 non-null  int64         
 1   acquisition_date    12000 non-null  datetime64[us]
 2   cancellation_month  7066 non-null   datetime64[us]
 3   gender              12000 non-null  str           
 4   marital_status      12000 non-null  str           
 5   age                 12000 non-null  int64         
 6   income_segment      11369 non-null  str           
 7   country             12000 non-null  str           
 8   channel             12000 non-null  str           
 9   campaign_id         12000 non-null  str           
 10  device_type         12000 non-null  str           
 11  plan_type           12000 non-null  str           
dtypes: datetime64[us](2), int64(2), str(8)
memory usage: 1.7 MB


In [7]:
df['gender'].value_counts(normalize=True)

gender
Female    0.501333
Male      0.498667
Name: proportion, dtype: float64

In [8]:
df['marital_status'].value_counts(normalize=True)

marital_status
Single     0.590583
Married    0.409417
Name: proportion, dtype: float64

In [9]:
df['income_segment'].value_counts(normalize=True)

income_segment
High       0.424224
Medium     0.375319
Low        0.143108
Premium    0.057349
Name: proportion, dtype: float64

In [10]:
df["acquisition_month"] = df["acquisition_date"].dt.to_period("M")

df[["user_id", "acquisition_date", "acquisition_month"]].head()

,user_id,acquisition_date,acquisition_month
0,1,2024-07-01,2024-07
1,2,2024-04-01,2024-04
2,3,2024-05-01,2024-05
3,4,2024-07-01,2024-07
4,5,2024-03-01,2024-03


In [11]:
acquisitions_by_cohort = (df['acquisition_date']
                          .value_counts()
                          .sort_index()
                          .reset_index())
acquisitions_by_cohort

,acquisition_date,count
0,2024-01-01,1502
1,2024-02-01,1528
2,2024-03-01,1465
3,2024-04-01,1509
4,2024-05-01,1541
5,2024-06-01,1493
6,2024-07-01,1500
7,2024-08-01,1462


In [12]:
fig = px.bar(acquisitions_by_cohort, 
       x = 'acquisition_date', 
       y='count')

fig.update_layout(
    yaxis_title = 'Number of Acquisitons',
    xaxis_title = 'Acquisition Date',
    title = 'Cohort Data'
)
fig.show()

In [13]:
df.head()

,user_id,acquisition_date,cancellation_month,gender,marital_status,age,income_segment,country,channel,campaign_id,device_type,plan_type,acquisition_month
0,1,2024-07-01,2025-02-01,Male,Married,31,Medium,Germany,Paid Ads,Paid Ads_C,iOS,Standard,2024-07
1,2,2024-04-01,2024-05-01,Male,Single,54,Premium,Netherlands,Referral,Referral_B,iOS,Standard,2024-04
2,3,2024-05-01,2024-07-01,Male,Single,34,Medium,Poland,Paid Ads,Paid Ads_A,Android,Standard,2024-05
3,4,2024-07-01,NaT,Male,Married,38,High,Belgium,Organic,Organic_C,Android,Standard,2024-07
4,5,2024-03-01,2024-04-01,Male,Single,25,Low,Sweden,Paid Ads,Paid Ads_A,Android,Basic,2024-03


In [40]:
def monthlydifferance(m1:str,m2:str,name:str):
    df[name] = np.where(
    df[m2].notna(),
    (
        (df[m2].dt.year - df[m1].dt.year) * 12
        + (df[m2].dt.month - df[m1].dt.month)
    ),
    np.nan
)


user_id	acquisition_date	acquisition_month	cancellation_month	month_churned
0	1	2024-07-01	2024-07	2025-02-01	**7.0**

(**2025-02-01** - **2024-07-01**) 

1. (2025 -  2024)*12 = 12
2. (2-7) = 5
3. (12 - 5) = **7**


In [41]:
m1 = 'acquisition_date'
m2 = 'cancellation_month'	
name = 'month_churned'

monthlydifferance(m1=m1,m2=m2,name=name)
df[["user_id", "acquisition_date", "acquisition_month",'cancellation_month',"month_churned"]].head()

,user_id,acquisition_date,acquisition_month,cancellation_month,month_churned
0,1,2024-07-01,2024-07,2025-02-01,7.0
1,2,2024-04-01,2024-04,2024-05-01,1.0
2,3,2024-05-01,2024-05,2024-07-01,2.0
3,4,2024-07-01,2024-07,NaT,NaN
4,5,2024-03-01,2024-03,2024-04-01,1.0


In [42]:
monthly_churn = df[["acquisition_month","month_churned"]].value_counts().sort_index().reset_index()
monthly_churn.head()

,acquisition_month,month_churned,count
0,2024-01,1.0,148
1,2024-01,2.0,95
2,2024-01,3.0,79
3,2024-01,4.0,76
4,2024-01,5.0,52


In [43]:
monthly_churn['month_churned'].value_counts()

month_churned
1.0     8
2.0     8
3.0     8
4.0     8
5.0     8
6.0     8
7.0     8
8.0     8
9.0     8
10.0    8
11.0    8
12.0    8
Name: count, dtype: int64

In [45]:
acquisitions_by_cohort["acquisition_month"] = acquisitions_by_cohort["acquisition_date"].dt.to_period("M")
acquisitions_by_cohort.head()

,acquisition_date,count,acquisition_month
0,2024-01-01,1502,2024-01
1,2024-02-01,1528,2024-02
2,2024-03-01,1465,2024-03
3,2024-04-01,1509,2024-04
4,2024-05-01,1541,2024-05


In [55]:
cohort_merge = monthly_churn.merge(acquisitions_by_cohort,on = 'acquisition_month', how = 'left')
# cohort_merge[]

In [47]:
cohort_churn = (
    df[df["month_churned"].notna()]
    .groupby(["acquisition_month", "month_churned"])
    .agg(users=("user_id", "count"))
    .reset_index()
)

cohort_churn["month_churned"] = cohort_churn["month_churned"].astype(int)

cohort_churn.head(15)

,acquisition_month,month_churned,users
0,2024-01,1,148
1,2024-01,2,95
2,2024-01,3,79
3,2024-01,4,76
4,2024-01,5,52
5,2024-01,6,49
6,2024-01,7,41
7,2024-01,8,37
8,2024-01,9,24
9,2024-01,10,26


In [48]:
cohort_size = (
    df.groupby("acquisition_month")
      .agg(new_users=("user_id", "count"))
      .reset_index()
)

cohort_size

,acquisition_month,new_users
0,2024-01,1502
1,2024-02,1528
2,2024-03,1465
3,2024-04,1509
4,2024-05,1541
5,2024-06,1493
6,2024-07,1500
7,2024-08,1462


In [49]:
max_month = 12
tenure_range = np.arange(1, max_month + 1)

tenure_range


array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12])

In [50]:
full_index = pd.MultiIndex.from_product(
    [cohort_size["acquisition_month"].unique(), tenure_range],
    names=["acquisition_month", "month_churned"]
)

full_index

cohort_data = (
    cohort_churn
    .set_index(["acquisition_month", "month_churned"])
    .reindex(full_index, fill_value=0)
    .reset_index()
)

cohort_data.head(15)

,acquisition_month,month_churned,users
0,2024-01,1,148
1,2024-01,2,95
2,2024-01,3,79
3,2024-01,4,76
4,2024-01,5,52
5,2024-01,6,49
6,2024-01,7,41
7,2024-01,8,37
8,2024-01,9,24
9,2024-01,10,26


In [56]:
cohort_merge.head()

,acquisition_month,month_churned,count_x,acquisition_date,count_y
0,2024-01,1.0,148,2024-01-01,1502
1,2024-01,2.0,95,2024-01-01,1502
2,2024-01,3.0,79,2024-01-01,1502
3,2024-01,4.0,76,2024-01-01,1502
4,2024-01,5.0,52,2024-01-01,1502


In [57]:
cohort_merge.rename(columns={'count_x':'churners','count_y':'cohort_size'},inplace=True)

cohort_merge.head(12)


,acquisition_month,month_churned,churners,acquisition_date,cohort_size
0,2024-01,1.0,148,2024-01-01,1502
1,2024-01,2.0,95,2024-01-01,1502
2,2024-01,3.0,79,2024-01-01,1502
3,2024-01,4.0,76,2024-01-01,1502
4,2024-01,5.0,52,2024-01-01,1502
5,2024-01,6.0,49,2024-01-01,1502
6,2024-01,7.0,41,2024-01-01,1502
7,2024-01,8.0,37,2024-01-01,1502
8,2024-01,9.0,24,2024-01-01,1502
9,2024-01,10.0,26,2024-01-01,1502


In [60]:
cohort_merge['churn_rate'] = cohort_merge['churners']/cohort_merge['cohort_size']
cohort_merge.head()

,acquisition_month,month_churned,churners,acquisition_date,cohort_size,churn_rate
0,2024-01,1.0,148,2024-01-01,1502,0.098535
1,2024-01,2.0,95,2024-01-01,1502,0.063249
2,2024-01,3.0,79,2024-01-01,1502,0.052597
3,2024-01,4.0,76,2024-01-01,1502,0.050599
4,2024-01,5.0,52,2024-01-01,1502,0.034621


In [61]:
#! WE MUST ALWAYS SORT THE VALUES REGARDLESS OF THE FACT WHETHER IT HAS BEEN SORTED OR NOT
cohort_merge = cohort_merge.sort_values(["acquisition_month", "month_churned"])
# -------------------------------
cohort_merge["cumulative_churn"] = (
    cohort_merge.groupby("acquisition_month")["churners"].cumsum()
)

cohort_merge.head()

,acquisition_month,month_churned,churners,acquisition_date,cohort_size,churn_rate,cumulative_churn
0,2024-01,1.0,148,2024-01-01,1502,0.098535,148
1,2024-01,2.0,95,2024-01-01,1502,0.063249,243
2,2024-01,3.0,79,2024-01-01,1502,0.052597,322
3,2024-01,4.0,76,2024-01-01,1502,0.050599,398
4,2024-01,5.0,52,2024-01-01,1502,0.034621,450


In [63]:
cohort_merge['active_users'] = cohort_merge['cohort_size'] - cohort_merge['cumulative_churn']
cohort_merge.head()


,acquisition_month,month_churned,churners,acquisition_date,cohort_size,churn_rate,cumulative_churn,active_users
0,2024-01,1.0,148,2024-01-01,1502,0.098535,148,1354
1,2024-01,2.0,95,2024-01-01,1502,0.063249,243,1259
2,2024-01,3.0,79,2024-01-01,1502,0.052597,322,1180
3,2024-01,4.0,76,2024-01-01,1502,0.050599,398,1104
4,2024-01,5.0,52,2024-01-01,1502,0.034621,450,1052


In [ ]:
cohort_merge['retention_rate'] = cohort_merge['active_users']/cohort_merge['cohort_size']

cohort_merge.head()



,acquisition_month,month_churned,churners,acquisition_date,cohort_size,churn_rate,cumulative_churn,active_users,retention_rate
0,2024-01,1.0,148,2024-01-01,1502,0.098535,148,1354,0.901465
1,2024-01,2.0,95,2024-01-01,1502,0.063249,243,1259,0.838216
2,2024-01,3.0,79,2024-01-01,1502,0.052597,322,1180,0.785619
3,2024-01,4.0,76,2024-01-01,1502,0.050599,398,1104,0.735020
4,2024-01,5.0,52,2024-01-01,1502,0.034621,450,1052,0.700399


In [71]:
cohort_merge['month_churned'] = cohort_merge['month_churned'].astype(int)
cohort_merge['acquisition_month'] = cohort_merge['acquisition_month'].astype(str) # պարտադիր է միայն կոնկրետ այս վիզուալիզացիայի համար

In [81]:
for_heatmap = cohort_merge.pivot(
    index = 'acquisition_month',
    columns = 'month_churned',
    values = 'cumulative_churn' # այստեղի փոփոխականը ենթակա է փոխարինման , միջին կամ մեդիան եկամուտ ըստ կոհորտի
)
for_heatmap

month_churned,1,2,3,4,5,6,7,8,9,10,11,12
acquisition_month,,,,,,,,,,,,
2024-01,148,243,322,398,450,499,540,577,601,627,638,651
2024-02,155,265,355,418,480,535,568,600,631,658,683,702
2024-03,511,741,845,909,959,988,1017,1038,1049,1065,1069,1078
2024-04,536,774,892,950,993,1023,1044,1059,1075,1086,1096,1103
2024-05,377,611,736,821,893,949,981,1009,1035,1055,1070,1079
2024-06,382,624,786,882,936,978,1018,1046,1064,1089,1104,1116
2024-07,142,265,340,417,468,523,559,606,630,658,674,691
2024-08,141,232,324,390,449,495,529,569,601,616,633,646


In [86]:
# փորձեք սարել ֆունկիցա , որը կվերցնի տարբեր տվյալներ
fig = px.imshow(
    img = for_heatmap,
    aspect = 'auto',
    color_continuous_scale = 'Blues',
    text_auto= '',
    labels = {
        'x': '',
        'y': '',
    },
    title = 'Cohort Analysis |Rentention Rate Per Month'
)

fig.update_xaxes(
    tickmode="array",
    tickvals=list(for_heatmap.columns),
    ticktext=[str(x) for x in for_heatmap.columns],
    side="top"
)

fig.update_yaxes(
    tickmode="array",
    tickvals=list(for_heatmap.index),
    ticktext=[str(y) for y in for_heatmap.index],
    autorange="reversed"
)

fig.show()